# GBIF quickstart — species occurrences over a bounding box

Fetch a year of **bird** occurrences over a small bounding box through the `EarthLens` facade, then map the occurrence points.

GBIF is a *vector* backend: `download()` returns a [pyramids](https://github.com/serapeum-org/pyramids) `FeatureCollection` (a `geopandas.GeoDataFrame` subclass) of `Point` geometries in `EPSG:4326`. Search is anonymous, so no credentials are needed.

> **Install.** GBIF search needs the `pygbif` SDK: `pip install earthlens[gbif]`. The query is paginated (300 records/page); `max_records` caps how many are pulled.

## Setup

Import `matplotlib` for the map and the unified `EarthLens` entry point.

In [ ]:
import matplotlib.pyplot as plt

from earthlens.core import EarthLens

## Request parameters

A small London bounding box and a one-year window. `variables=["birds"]` resolves to GBIF backbone `taxonKey` 212; you can also pass a raw key (`[212]`) or a name lookup (`["taxon:Panthera leo"]`).

In [ ]:
bbox_lat = [51.4, 51.6]  # [lat_min, lat_max]
bbox_lon = [-0.2, 0.0]  # [lon_min, lon_max]
start, end = "2020-01-01", "2020-12-31"

## Download

One call returns the occurrence `FeatureCollection` and writes a GeoParquet under `path`.

In [ ]:
fc = EarthLens(
    data_source="gbif",
    variables=["birds"],
    start=start,
    end=end,
    lat_lim=bbox_lat,
    lon_lim=bbox_lon,
    path="out/gbif",
    max_records=2000,
).download()

print(f"{len(fc)} occurrences")
fc[["scientificName", "eventDate", "license"]].head()

## Map the occurrences

Plot the points; the geometry column is ready for any geopandas workflow.

In [ ]:
ax = fc.plot(markersize=4, color="darkgreen")
ax.set_title("GBIF bird occurrences — London, 2020")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
plt.show()

## Licensing

GBIF records carry per-dataset licenses (`CC0` / `CC-BY` / `CC-BY-NC`). When any record is `CC-BY-NC`, the backend raises a `LicenseWarning` — honour the non-commercial obligation before redistributing.